# CSV에서 channel_id 추출 (YT_ChannelData)

vling에서 직접 다운로드한 `YT_ChannelData_2026-03-28.csv`의  
`Channel Link` 컬럼에서 YouTube `channel_id`를 추출해 새 컬럼으로 추가합니다.

## 원본 파일 구조

| 컬럼 | 예시 |
|---|---|
| `Channel Name` | 다이어트 과학자 최겸 Gyum Choi |
| `Channel Link` | `https://vling.net/channel/UChXDq9Izwq-pTzeUWzwXL3A` |
| `Subscribers` | 603000 |
| `Average Daily Views` | 79982 |

→ `Channel Link`의 `/channel/{CHANNEL_ID}` 에서 `UChXDq9Izwq-pTzeUWzwXL3A` 추출

In [ ]:
import re
import pandas as pd
from pathlib import Path

CHANNEL_DATA_CSV = Path("YT_ChannelData_2026-03-28.csv")

df_raw = pd.read_csv(CHANNEL_DATA_CSV, encoding="utf-8-sig")

print(f"로드 완료: {len(df_raw)}개 채널")
print(f"컬럼: {df_raw.columns.tolist()}")
df_raw.head(5)

### channel_id 추출

`Channel Link` 컬럼의 URL에서 `UC`로 시작하는 채널 ID를 파싱합니다.

In [ ]:
_ID_RE = re.compile(r"/(UC[\w-]+)")

df_clean = df_raw.copy()

# channel_id 추출
df_clean["channel_id"] = (
    df_clean["Channel Link"]
    .fillna("")
    .apply(lambda url: m.group(1) if (m := _ID_RE.search(url)) else "")
)

# 컬럼 정리 및 이름 통일
df_clean = df_clean.rename(columns={
    "Channel Name":        "channel_name",
    "Subscribers":         "subscribers",
    "Average Daily Views": "avg_daily_views",
})
df_clean = df_clean[["channel_name", "channel_id", "subscribers", "avg_daily_views"]]

# 파싱 실패 확인
failed = df_clean[df_clean["channel_id"] == ""]
print(f"channel_id 추출 성공: {(df_clean['channel_id'] != '').sum()} / {len(df_clean)}")
if not failed.empty:
    print(f"추출 실패 ({len(failed)}개):")
    print(failed[["channel_name"]].to_string(index=False))

df_clean.head(10)

### 결과 저장

`channel_id`가 있는 채널만 남겨 정리된 CSV로 저장합니다.

In [ ]:
OUTPUT_CSV = Path("YT_ChannelData_2026-03-28_clean.csv")

df_result = df_clean[df_clean["channel_id"] != ""].reset_index(drop=True)
df_result.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"저장 완료: {len(df_result)}개 채널 → {OUTPUT_CSV}")
print(f"channel_id 없어서 제외: {len(df_clean) - len(df_result)}개")
df_result.head(10)